# Reciprocal Rank Fusion (RRF) — Merging Two Ranked Lists

BM25 gives one ranking. Semantic gives another. How do you combine them?

**RRF formula: score(doc) = sum( 1 / (k + rank_i) ) for each ranker i**

This notebook demonstrates why RRF works and what `k` controls.

In [ ]:
# Simulated rankings from BM25 and Semantic
bm25_ranking = ["doc_A", "doc_C", "doc_B", "doc_D", "doc_E"]
semantic_ranking = ["doc_B", "doc_A", "doc_D", "doc_C", "doc_F"]

print("BM25 ranking:    ", bm25_ranking)
print("Semantic ranking:", semantic_ranking)

In [ ]:
def rrf_score(doc, rankings, k=60):
    """Compute RRF score for a document across multiple rankings."""
    score = 0
    details = []
    for ranking_name, ranking in rankings.items():
        if doc in ranking:
            rank = ranking.index(doc) + 1  # 1-indexed
            contribution = 1 / (k + rank)
            score += contribution
            details.append(f"{ranking_name}: rank {rank} → 1/({k}+{rank}) = {contribution:.6f}")
        else:
            details.append(f"{ranking_name}: not ranked")
    return score, details

# Compute RRF for all unique documents
all_docs = list(set(bm25_ranking + semantic_ranking))
rankings = {"BM25": bm25_ranking, "Semantic": semantic_ranking}

print(f"RRF scores (k=60):\n")
rrf_results = []
for doc in all_docs:
    score, details = rrf_score(doc, rankings, k=60)
    rrf_results.append((doc, score, details))

rrf_results.sort(key=lambda x: x[1], reverse=True)

for doc, score, details in rrf_results:
    print(f"{doc}: RRF = {score:.6f}")
    for d in details:
        print(f"    {d}")
    print()

In [ ]:
# What does k control?
print("Effect of k on RRF scoring:\n")
print("Higher k = more equal weighting between ranks")
print("Lower k = rank #1 dominates more\n")

for k in [1, 10, 60, 200]:
    scores = []
    for doc in ["doc_A", "doc_B"]:
        score, _ = rrf_score(doc, rankings, k=k)
        scores.append((doc, score))
    scores.sort(key=lambda x: x[1], reverse=True)
    print(f"k={k:>3}: Winner = {scores[0][0]} ({scores[0][1]:.6f}) vs {scores[1][0]} ({scores[1][1]:.6f})")

## Key Takeaways

1. **RRF rewards consistency** — a doc ranked #2 in BOTH lists beats a doc ranked #1 in one and #10 in another
2. **k=60 is the standard** — used in the original RRF paper, works well in practice
3. **No score normalization needed** — RRF only uses rank positions, not raw scores
4. **Works with any number of rankers** — can fuse 2, 3, or more ranked lists